In [41]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import xgboost as xgb

X_train = pd.read_csv("data/X_train_purged_df.csv", sep=',')
mapping = {"Absence" : 0, "Presence" : 1}
X_train["Heart Disease"] = X_train["Heart Disease"].map(mapping)
y = X_train["Heart Disease"]
X_train.drop(columns="Heart Disease", inplace=True)

X_test = pd.read_csv("data/X_test_purged_df.csv", sep=',')

In [42]:
print(X_train.head(1))

   num__Age  num__BP  num__Cholesterol  num__Max HR  num__ST depression  \
0  0.346801 -0.70096          -2.01926    -1.089147            0.510266   

   cat__Chest pain type_4  cat__Exercise angina_0  cat__Exercise angina_1  \
0                     0.0                     1.0                     0.0   

   cat__Slope of ST_1  cat__Number of vessels fluro_0  cat__Thallium_3  \
0                 0.0                             1.0              1.0   

   cat__Thallium_7  
0              0.0  


In [43]:
print(y.head())

0    0
1    1
2    1
3    1
4    0
Name: Heart Disease, dtype: int64


In [44]:
xgb_model = xgb.XGBClassifier(
    n_estimators=1000,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    colsample_bylevel=0.8,
    early_stopping_rounds=50,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

X_train, X_val, y_train, y_val = train_test_split(X_train, y, test_size=0.2, random_state=42, stratify=y)

In [45]:
xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)])

y_pred_val = xgb_model.predict(X_val)

percentage = accuracy_score(y_val, y_pred_val)

print(f" Accuracy: {percentage*100}%")

[0]	validation_0-logloss:0.66134
[1]	validation_0-logloss:0.63468
[2]	validation_0-logloss:0.61051
[3]	validation_0-logloss:0.58873
[4]	validation_0-logloss:0.56958
[5]	validation_0-logloss:0.55230
[6]	validation_0-logloss:0.53641
[7]	validation_0-logloss:0.52010
[8]	validation_0-logloss:0.50523
[9]	validation_0-logloss:0.49269
[10]	validation_0-logloss:0.47990
[11]	validation_0-logloss:0.46860
[12]	validation_0-logloss:0.45737
[13]	validation_0-logloss:0.44680
[14]	validation_0-logloss:0.43702
[15]	validation_0-logloss:0.42818
[16]	validation_0-logloss:0.41998
[17]	validation_0-logloss:0.41292
[18]	validation_0-logloss:0.40555
[19]	validation_0-logloss:0.39850
[20]	validation_0-logloss:0.39183
[21]	validation_0-logloss:0.38598
[22]	validation_0-logloss:0.38017
[23]	validation_0-logloss:0.37469
[24]	validation_0-logloss:0.37000
[25]	validation_0-logloss:0.36517
[26]	validation_0-logloss:0.36072
[27]	validation_0-logloss:0.35653
[28]	validation_0-logloss:0.35274
[29]	validation_0-loglos

In [46]:
try:
    id = X_test.id
    X_test.drop(columns="id", inplace=True)
except AttributeError:
    print("Already dropped")
finally:
    print()
kaggle_pred = xgb_model.predict(X_test)

submit = pd.DataFrame({"id" : id,
                       "Heart Disease" : kaggle_pred})

filename = "data/submitXGBoost.csv"
submit.to_csv(filename, index=False)